# Planetary Computer — Monthly Change Detection

Pulls Sentinel-2 imagery for one AOI across two time windows, builds cloud-free median composites, and produces a simple change map (NDVI delta — picks up new construction, solar arrays, deforestation).

**No account needed.** Microsoft Planetary Computer is free and the STAC API is open. The `planetary_computer` package signs the asset URLs so you can read them directly.

**Default AOI:** Bhadla Solar Park, Rajasthan — the world's largest solar park, still expanding. Easy visible change. Swap `bbox` for your own AOI.

## 1. Install dependencies
Run once. Skip if you've already got these.

In [ ]:
%pip install -q pystac-client planetary-computer odc-stac rioxarray matplotlib numpy

## 2. Configure AOI and date windows

In [ ]:
import pystac_client
import planetary_computer
import odc.stac
import numpy as np
import matplotlib.pyplot as plt

# AOI = bbox [min_lon, min_lat, max_lon, max_lat]
# Bhadla Solar Park, Rajasthan, India
bbox = [71.85, 27.45, 71.95, 27.55]

# Two windows to compare. Wider window = more scenes = better median composite.
window_a = "2019-01-01/2019-03-31"   # before
window_b = "2024-01-01/2024-03-31"   # after

# Cloud-cover threshold for scene selection
max_cloud = 10

## 3. Connect to Planetary Computer STAC and search

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def search(window):
    return catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=window,
        query={"eo:cloud_cover": {"lt": max_cloud}},
    ).item_collection()

items_a = search(window_a)
items_b = search(window_b)
print(f"Window A ({window_a}): {len(items_a)} scenes")
print(f"Window B ({window_b}): {len(items_b)} scenes")

## 4. Load bands and build median composites
Median across time dimension = cloud-free composite without you having to mask anything.

In [ ]:
bands = ["B02", "B03", "B04", "B08"]  # blue, green, red, NIR

def composite(items):
    ds = odc.stac.load(
        items,
        bbox=bbox,
        bands=bands,
        resolution=10,
        chunks={},  # lazy
    )
    return ds.median(dim="time").compute()

comp_a = composite(items_a)
comp_b = composite(items_b)
print("Composite A shape:", comp_a.B04.shape)
print("Composite B shape:", comp_b.B04.shape)

## 5. Visualize RGB side by side

In [ ]:
def rgb(comp):
    arr = np.stack([comp.B04.values, comp.B03.values, comp.B02.values], axis=-1)
    return np.clip(arr / 3000.0, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(rgb(comp_a)); axes[0].set_title(f"Before — {window_a}"); axes[0].axis("off")
axes[1].imshow(rgb(comp_b)); axes[1].set_title(f"After — {window_b}"); axes[1].axis("off")
plt.tight_layout(); plt.show()

## 6. Change detection — NDVI delta
NDVI drops when vegetation is replaced by solar panels, buildings, or bare ground. Red pixels = vegetation loss / new construction. Blue = greening.

In [ ]:
def ndvi(comp):
    nir = comp.B08.astype("float32")
    red = comp.B04.astype("float32")
    return ((nir - red) / (nir + red + 1e-6)).values

ndvi_a = ndvi(comp_a)
ndvi_b = ndvi(comp_b)
diff = ndvi_b - ndvi_a

plt.figure(figsize=(10, 8))
im = plt.imshow(diff, cmap="RdBu", vmin=-0.5, vmax=0.5)
plt.colorbar(im, label="NDVI change (after - before)")
plt.title("Change map — red = vegetation loss / new build, blue = greening")
plt.axis("off"); plt.show()

## 7. Where to go next

- **Monthly cadence:** loop over a list of monthly windows (`2024-01`, `2024-02`, ...) and store one composite per month, then diff consecutive pairs.
- **Different change signal:** for solar panels specifically, SWIR (`B11`, `B12`) is more discriminative than NDVI — panels are bright in SWIR. For buildings, try NDBI = (B11 - B08) / (B11 + B08).
- **Threshold + count:** `(diff < -0.3).sum() * 100` gives square meters of significant vegetation loss.
- **Other collections:** swap `sentinel-2-l2a` for `landsat-c2-l2` (30 m, longer history back to 1984) or `hls2-l30` (harmonized Landsat+S2).
- **Google Earth Engine:** the same workflow in GEE is ~15 lines of JavaScript. Planetary Computer wins if you prefer Python + your own compute; GEE wins if you want server-side reductions and don't mind their environment.